In [ ]:
 # IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
#  RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

super_ai_engineer_ss_6_thai_language_image_captioning_path = kagglehub.competition_download('super-ai-engineer-ss-6-thai-language-image-captioning')

print('Data source import complete.')


In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers
!pip install -q accelerate
!pip install -q -i https://pypi.org/simple/ bitsandbytes
!pip install -q pythainlp fairseq sacremoses

In [ ]:
# 1. ลดเวอร์ชั่นของตัวติดตั้ง (pip) ให้อยู่ต่ำกว่า 24.1 เพื่อลดความเข้มงวด
!python -m pip install -q "pip<24.1"

# 2. ติดตั้งแพ็กเกจแปลภาษาและจัดการข้อความที่ต้องการอีกครั้ง
!pip install -q pythainlp fairseq sacremoses

# 3. ตรวจสอบว่าไลบรารีถูกติดตั้งเรียบร้อยแล้วหรือไม่
import pythainlp
print("ติดตั้ง PyThaiNLP สำเร็จ! เวอร์ชั่น:", pythainlp.__version__)

In [ ]:
import torch
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
from pythainlp.translate import Translate
import pandas as pd

In [ ]:
# สร้างกฎการย่อส่วนโมเดลให้เหลือ 4-bit เพื่อประหยัดพื้นที่ GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
# 1. โหลดเพื่อนชาวต่างชาติ (BLIP-2) พร้อมประกอบร่างกับกฎการย่อส่วน 4-bit
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-6.7b-coco")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-6.7b-coco",
    quantization_config=bnb_config,
    device_map='cuda', # จับใส่ GPU
    torch_dtype=torch.float16
)



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from PIL import Image
from pathlib import Path

# 1. โหลดล่ามแปลภาษา NLLB "แบบไม่ผ่านคนกลาง"
print("กำลังโหลด Tokenizer และ Model ของ NLLB...")
model_name = "facebook/nllb-200-distilled-600M"

# กำหนดให้รับภาษาอังกฤษ (eng_Latn) เป็นหลัก
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn")
translator_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ย้ายโมเดลล่ามไปไว้บน GPU เพื่อความเร็ว
translator_model = translator_model.to("cuda")

# 2. ปรับฟังก์ชันให้ใช้ล่ามตัวใหม่
def process_image(path: Path, is_coco: bool):
    image = Image.open(path)
    image_id = str(path.relative_to(path.parents[1 if is_coco else 2]).with_suffix(''))

    # 2.1 ให้ BLIP-2 บรรยายเป็นภาษาอังกฤษ
    inputs_img = processor(image, return_tensors="pt").to("cuda", torch.float16)
    out_img = model.generate(**inputs_img, max_new_tokens=30)
    caption_en = processor.decode(out_img[0], skip_special_tokens=True)

    # 2.2 นำข้อความภาษาอังกฤษมาหั่นเป็นตัวเลข (Tokenize)
    inputs_text = tokenizer(caption_en, return_tensors="pt").to("cuda")

    # 2.3 สั่งให้โมเดลแปล โดยบังคับภาษาปลายทางเป็น ภาษาไทย (tha_Thai)
    thai_lang_id = tokenizer.lang_code_to_id["tha_Thai"]
    translated_tokens = translator_model.generate(
        **inputs_text,
        forced_bos_token_id=thai_lang_id, # บังคับให้เริ่มประโยคด้วยแท็กภาษาไทย
        max_new_tokens=30
    )

    # 2.4 แปลงตัวเลขกลับมาเป็นข้อความภาษาไทย
    caption_th = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

    return (image_id, caption_th)

In [ ]:
def process_image(path: Path, is_coco: bool):
    # 3.1 เปิดรูปภาพขึ้นมา
    image = Image.open(path)
    image_id = str(path.relative_to(path.parents[1 if is_coco else 2]).with_suffix(''))

    # 3.2 แปลงรูปภาพเป็นตัวเลขส่งให้ BLIP-2
    inputs = processor(image, return_tensors="pt").to("cuda", torch.float16)

    # 3.3 BLIP-2 สร้างประโยคบรรยายภาพเป็นภาษาอังกฤษ
    out = model.generate(**inputs)
    caption_en = processor.decode(out[0], skip_special_tokens=True)

    # 3.4 ส่งประโยคอังกฤษให้ล่ามแปลเป็นภาษาไทย
    caption_th = translate_model.translate(caption_en)

    return (image_id, caption_th)

In [ ]:
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import torch

# 1. โหลดกระดาษคำตอบ โดยบังคับให้คอลัมน์ id เป็น String ป้องกัน Pandas ตัดเลข 0
submission_df = pd.read_csv("/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/sample_submission.csv", dtype=str)

# 2. กำหนดโฟลเดอร์ (อ้างอิงจากเส้นทางของคุณ)
test_dirs = [
    Path('/kaggle/input/coco-2017-dataset/coco2017/test2017'),
    Path('/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/train/train/food'),
    Path('/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel')
]

# 3. สร้าง "สมุดหน้าเหลือง" (Dictionary Mapping)
print("กำลังสแกนหาไฟล์รูปภาพทั้งหมด...")
image_path_dict = {}

for directory in test_dirs:
    for img_path in directory.glob('*.jpg'):
        stem_name = img_path.stem # ดึงมาแค่ชื่อไฟล์ เช่น '000000001354' หรือ '00002'

        # เก็บทั้งชื่อเต็ม และชื่อที่ตัดเลข 0 ข้างหน้าออก (เผื่อไว้ทั้ง 2 กรณี)
        image_path_dict[stem_name] = img_path

        # แปลงเป็น int เพื่อตัด 0 นำหน้า แล้วแปลงกลับเป็น str เพื่อใช้เป็น Key
        stripped_name = str(int(stem_name))
        image_path_dict[stripped_name] = img_path

print(f"สแกนเสร็จสิ้น! พบไฟล์รูปภาพในระบบ {len(set(image_path_dict.values()))} รูป")

# 4. เริ่มกระบวนการวนลูปทำนาย
print("เริ่มดึงภาพและสร้างคำบรรยาย...")

for index, row in tqdm(submission_df.iterrows(), total=len(submission_df)):
    # *สำคัญ*: เช็คชื่อคอลัมน์ใน CSV ของคุณให้ดีว่าชื่อ 'image_id' หรือ 'id'
    # ถ้าชื่อคอลัมน์คือ id ให้เปลี่ยน row['image_id'] เป็น row['id']
    target_id = str(row['image_id'])

    # ค้นหาเส้นทางไฟล์จากสมุดหน้าเหลือง
    img_path = image_path_dict.get(target_id)

    if img_path:
        # ถ้ารูปหาเจอ ก็ส่งเข้ากระบวนการแปล (โค้ดเดิมของคุณ)
        image = Image.open(img_path)

        # -- กระบวนการ BLIP-2 --
        inputs_img = processor(image, return_tensors="pt").to("cuda", torch.float16)
        out_img = model.generate(**inputs_img, max_new_tokens=30)
        caption_en = processor.decode(out_img[0], skip_special_tokens=True)


        # -- กระบวนการ NLLB --
        inputs_text = tokenizer(caption_en, return_tensors="pt").to("cuda")

        # [จุดที่ต้องแก้] ใช้ฟังก์ชันมาตรฐานในการดึง ID ของภาษาไทย
        thai_lang_id = tokenizer.convert_tokens_to_ids("tha_Thai")

        translated_tokens = translator_model.generate(
            **inputs_text,
            forced_bos_token_id=thai_lang_id,
            max_new_tokens=30
        )
        caption_th = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

        # บันทึกคำตอบ
        submission_df.at[index, 'caption'] = caption_th
    else:
        # ถ้ายังหาไม่เจออีก จะได้รู้ว่า ID อะไรที่มีปัญหา
        print(f"แปลกมาก! หารูปภาพ ID {target_id} ไม่เจอในโฟลเดอร์ไหนเลย")
        submission_df.at[index, 'caption'] = "ไม่พบรูปภาพ"

# 5. บันทึกไฟล์ส่ง
submission_df.to_csv('submission.csv', index=False)
print("รันเสร็จสมบูรณ์ พร้อมส่ง Kaggle!")

In [ ]:
import pandas as pd
from pythainlp.tokenize import word_tokenize
from collections import Counter
import matplotlib.pyplot as plt

def analyze_translation_errors(submission_csv_path):
    # 1. โหลดข้อมูล CSV ที่รันเสร็จแล้วเข้ามา
    df = pd.read_csv(submission_csv_path)

    # 2. การวิเคราะห์ความยาวประโยค (Length Analysis)
    # ใช้ PyThaiNLP นับจำนวนคำในแต่ละประโยค
    df['word_count'] = df['caption'].astype(str).apply(lambda x: len(word_tokenize(x)))

    print("--- 📊 การวิเคราะห์ความยาวประโยค ---")
    print(f"ประโยคที่สั้นที่สุดมี {df['word_count'].min()} คำ")
    print(f"ประโยคที่ยาวที่สุดมี {df['word_count'].max()} คำ (ระวัง! ถ้ายาวไปอาจจะเกิดอาการหลอนวนลูป)")

    # 3. กระบวนการหาคำที่โมเดลแปลบ่อยผิดปกติ (Word Frequency)
    all_words = []
    for caption in df['caption'].dropna():
        words = word_tokenize(str(caption))
        # กรองเอาเฉพาะคำที่มีความยาว ไม่เอาพวกช่องว่าง
        all_words.extend([w for w in words if len(w.strip()) > 0])

    top_words = Counter(all_words).most_common(10)
    print("\n--- 🔍 10 คำที่โมเดล NLLB ชอบใช้แปลมากที่สุด ---")
    for word, count in top_words:
        print(f"คำว่า '{word}': {count} ครั้ง")

# เรียกใช้งานฟังก์ชัน (ใส่ path ไฟล์ submission.csv ของคุณ)


In [ ]:
analyze_translation_errors("submission.csv")

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig

model_id = "typhoon-ai/llama-3-typhoon-v1.5-8b-vision-preview"

# 1. โหลด Config และแก้ปัญหา pad_token_id
print("Loading config...")
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
if not hasattr(config, "pad_token_id") or config.pad_token_id is None:
    config.pad_token_id = config.eos_token_id if config.eos_token_id is not None else 0

# 2. โหลด Processor
print("Loading processor...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)



In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image

# ใช้ Typhoon 2 Vision (ใหม่ล่าสุดและเสถียรกว่า)
model_id = "typhoon-ai/typhoon2-qwen2vl-7b-vision-instruct"

print("Loading Processor...")
processor = AutoProcessor.from_pretrained(model_id)

print("Loading Model... (Typhoon 2)")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto", # รุ่นนี้รองรับ auto ได้ปกติ ไม่บั๊กเหมือนรุ่นเก่า
    trust_remote_code=True
)

print("Model Loaded Successfully! 🎉")

# --- ตัวอย่างการรันทำนายภาพ ---
def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "อธิบายรูปภาพนี้เป็นภาษาไทย"},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=128)
    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
    return output_text[0]

Loading Processor...


preprocessor_config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model Loaded Successfully! 🎉


In [ ]:
# พิมพ์ดูประโยคเฉลยของรูปแรก
print(train_data["coco/train2017/000000373716.jpg"])

['ผู้หญิงสวมเสื้อแขนยาวสีขาวและเด็กนั่งเล่นกับสุนัขอยู่ ในสวนหย่อม', 'สาวคนนึงกำลังพาเด็กมานั่งเล่นอยู่ภายในสนามหญ้าพร้อมกับสุนัข', 'ภาพขาวดำ ผู้หญิงนั่งบนพื้นอุ้มเด็กบนตัก ข้าง ๆ มีหมาสองตัว ด้านหลังมีม้านั่งไม้']


In [ ]:
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 21.1 MB/s eta 0:00:00:00:0100:01
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [ ]:
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
import torch
from qwen_vl_utils import process_vision_info

# ==========================================
# 1. ตั้งค่า Path โฟลเดอร์ (แก้ชื่อ Dataset ให้ตรง)
# ==========================================
DATA_DIR = "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning"
TEST_DIR = os.path.join(DATA_DIR, "test")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/sample_submission.csv")
OUTPUT_FILE = "submission222.csv"

# ==========================================
# 2. เตรียมรายการไฟล์ Test จาก sample_submission
# ==========================================
sub_df = pd.read_csv(SAMPLE_SUB_PATH)

test_image_paths = {}
for root, dirs, files in os.walk(TEST_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            # โค้ดนี้จะดำน้ำลงไปหาไฟล์ในโฟลเดอร์ food และ travel ให้โดยอัตโนมัติ
            test_image_paths[file] = os.path.join(root, file)

# ==========================================
# 3. ฟังก์ชันสำหรับทำนายภาพ
# ==========================================
def generate_thai_caption(image_path):
    image = Image.open(image_path).convert("RGB")

    # 🌟 ปรับ Prompt ใหม่ให้โมเดลเค้นรายละเอียดออกมาให้หมด คล้ายเฉลย
    prompt_text = (
        "บรรยายรายละเอียดทั้งหมดในภาพนี้เป็นภาษาไทย 1 ประโยคแบบละเอียด "
        "โดยระบุให้ครบถ้วนว่า ใคร ทำอะไร ที่ไหน มีลักษณะอย่างไร (เช่น สีเสื้อ ท่าทาง สัตว์ สิ่งของรอบตัว ฉากหลัง) "
        "หากเป็นภาพขาวดำให้ระบุด้วยว่าเป็นภาพขาวดำ"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        # เพิ่ม max_new_tokens เป็น 80 เผื่อโมเดลบรรยายยาว และปรับ temperature ให้เป็น 0.2 เพื่อลดการแต่งเรื่องมั่ว
        generated_ids = model.generate(**inputs, max_new_tokens=80, temperature=0.2)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
    return output_text.strip()

# ==========================================
# 4. เริ่มรันทำนายภาพทั้งหมด
# ==========================================
print(f"กำลังเริ่มประมวลผล {len(sub_df)} รูปภาพ...")

# ดึงชื่อคอลัมน์แรกจาก sample_submission (มักจะเป็น 'image_id' หรือ 'id')
id_col = sub_df.columns[0]
caption_col = sub_df.columns[1]

predictions = []

for index, row in tqdm(sub_df.iterrows(), total=len(sub_df)):
    image_filename = str(row[id_col])

    # บางทีในไฟล์ csv ไม่มีนามสกุลไฟล์แถมมาให้ หรือเป็น path เต็ม เราพยายามหาส่วนที่เป็นชื่อไฟล์
    search_name = os.path.basename(image_filename)

    if search_name in test_image_paths:
        img_path = test_image_paths[search_name]
        try:
            caption = generate_thai_caption(img_path)
        except Exception as e:
            print(f"Error ({search_name}): {e}")
            caption = "ไม่มีคำอธิบาย"
    else:
        # กรณีหาไฟล์รูปไม่เจอ
        caption = "ไม่มีคำอธิบาย"

    predictions.append(caption)

# ==========================================
# 5. บันทึกผลลัพธ์
# ==========================================
sub_df[caption_col] = predictions
# เปลี่ยนจากแบบเดิม เป็นเพิ่ม encoding เข้าไป
sub_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f"\n🎉 เสร็จเรียบร้อย! สร้างไฟล์ {OUTPUT_FILE} สำหรับส่งคำตอบแล้ว")

กำลังเริ่มประมวลผล 2000 รูปภาพ...


100%|██████████| 2000/2000 [00:00<00:00, 27944.79it/s]


🎉 เสร็จเรียบร้อย! สร้างไฟล์ submission222.csv สำหรับส่งคำตอบแล้ว


In [ ]:
# --- โค้ดสำหรับรันเช็คอาการ Error 1 รูป ---
print("--- เริ่มการวิเคราะห์ปัญหา ---")

# ดึงข้อมูลแถวแรกสุดจาก sample_submission
id_col = sub_df.columns[0]
row = sub_df.iloc[0]
image_filename = str(row[id_col])

# 1. เช็คและจัดการนามสกุลไฟล์
if not image_filename.lower().endswith(('.jpg', '.png', '.jpeg')):
    search_name = image_filename + ".jpg" # ลองเติม .jpg ให้
else:
    search_name = os.path.basename(image_filename)

print(f"1. ชื่อไฟล์ที่ต้องการหาในระบบคือ: '{search_name}'")

# 2. เช็คว่าเจอไฟล์ไหม
if search_name in test_image_paths:
    img_path = test_image_paths[search_name]
    print(f"2. สถานะ: ✅ เจอไฟล์รูปภาพแล้ว! อยู่ที่ '{img_path}'")

    print("3. กำลังส่งให้โมเดลทำนาย... (ถ้า Error จะขึ้นสีแดงด้านล่างนี้เลย)")
    # รันแบบไม่มี try-except เพื่อดึง Error สีแดงของจริงออกมาดู
    caption = generate_thai_caption(img_path)
    print(f"4. สถานะ: ✅ โมเดลทำนายสำเร็จ!\nผลลัพธ์คือ: {caption}")

else:
    print(f"2. สถานะ: ❌ หาไฟล์รูปไม่เจอ!")
    print("\nลองสุ่มรายชื่อไฟล์ที่มีในโฟลเดอร์ test มาให้ดู 5 รูป:")
    print(list(test_image_paths.keys())[:5])
    print("\nวิธีแก้: ชื่อไฟล์ใน sample_submission ไม่ตรงกับชื่อไฟล์จริง ลองดูตัวอย่างชื่อไฟล์ด้านบนว่ามันต่างกันตรงไหน (เช่น มีโฟลเดอร์นำหน้า หรือนามสกุลไฟล์ต่างกัน)")

--- เริ่มการวิเคราะห์ปัญหา ---
1. ชื่อไฟล์ที่ต้องการหาในระบบคือ: '1354.jpg'
2. สถานะ: ❌ หาไฟล์รูปไม่เจอ!

ลองสุ่มรายชื่อไฟล์ที่มีในโฟลเดอร์ test มาให้ดู 5 รูป:
['00767.jpg', '00266.jpg', '01496.jpg', '01600.jpg', '00847.jpg']

วิธีแก้: ชื่อไฟล์ใน sample_submission ไม่ตรงกับชื่อไฟล์จริง ลองดูตัวอย่างชื่อไฟล์ด้านบนว่ามันต่างกันตรงไหน (เช่น มีโฟลเดอร์นำหน้า หรือนามสกุลไฟล์ต่างกัน)


In [ ]:
# ==========================================
# 4. เริ่มรันทำนายภาพทั้งหมด (เวอร์ชันแก้ปัญหาชื่อไฟล์)
# ==========================================
print(f"กำลังเริ่มประมวลผล {len(sub_df)} รูปภาพ...")

id_col = sub_df.columns[0]
caption_col = sub_df.columns[1]

predictions = []

for index, row in tqdm(sub_df.iterrows(), total=len(sub_df)):
    raw_id = str(row[id_col]).replace('.jpg', '') # ดึงตัวเลขมา และลบ .jpg ออกเผื่อมีติดมา

    # 🌟 กุญแจสำคัญ: แปลงเป็นตัวเลขแล้วเติม 0 ข้างหน้าให้ครบ 5 หลัก
    search_name = f"{int(raw_id):05d}.jpg"

    if search_name in test_image_paths:
        img_path = test_image_paths[search_name]
        try:
            caption = generate_thai_caption(img_path)
        except Exception as e:
            print(f"Error ({search_name}): {e}")
            caption = "ไม่มีคำอธิบาย"
    else:
        # เผื่อกรณีฉุกเฉิน
        print(f"หาไฟล์ไม่เจอ: {search_name}")
        caption = "ไม่มีคำอธิบาย"

    predictions.append(caption)

# ==========================================
# 5. บันทึกผลลัพธ์ (อย่าลืม utf-8-sig เพื่อให้เปิดใน Excel ได้ไม่เป็นต่างด้าว)
# ==========================================
sub_df[caption_col] = predictions
sub_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f"\n🎉 เสร็จเรียบร้อย! สร้างไฟล์ {OUTPUT_FILE} สำหรับส่งคำตอบแล้ว")

กำลังเริ่มประมวลผล 2000 รูปภาพ...


100%|██████████| 2000/2000 [1:38:08<00:00,  2.94s/it]


🎉 เสร็จเรียบร้อย! สร้างไฟล์ submission222.csv สำหรับส่งคำตอบแล้ว
